In [1]:
import sys
import pickle
import numpy as np
import pandas as pd
from itertools import combinations
from math import factorial
from collections import defaultdict
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly, PROCESSED_DATA_PATH

CHANNELS: dict[str, tuple[str]] = {
  "Domestic Baseline": (Sources.cpi_inflation, Sources.IPI),
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.UST_10Y, Sources.HY_OAS),
  "FX Pass-Through": (Sources.USDMYR,),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

def shapley_r2(y: pd.Series, X: pd.DataFrame) -> dict[str, float]:
  predictors = list(X.columns)
  p = len(predictors)
  Xc = sm.add_constant(X)
  r2_cache: dict[frozenset, float] = {frozenset(): 0.0}
  for k in range(1, p + 1):
    for combo in combinations(predictors, k):
      r2_cache[frozenset(combo)] = sm.OLS(y, Xc[["const"] + list(combo)]).fit().rsquared
  shapley = {}
  for j in predictors:
    rest = [v for v in predictors if v != j]
    total = 0.0
    for k in range(len(rest) + 1):
      weight = factorial(k) * factorial(p - k - 1) / factorial(p)
      for combo in combinations(rest, k):
        s = frozenset(combo)
        total += weight * (r2_cache[s | {j}] - r2_cache[s])
    shapley[j] = total
  return shapley

full_models: dict[str, RegressionResultsWrapper] = {}
fit_dict, partialr2_dict = {}, {}
wald_dict, beta_ci_dict = defaultdict(dict), defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full
  ci = m_full.conf_int(alpha=0.1)

  fit_dict[d_var] = {
    ("Full R2"): m_full.rsquared,
    ("Adjusted R2"): m_full.rsquared_adj
  }

  shap = shapley_r2(data[d_var], data[ALL_PREDICTORS])
  partialr2_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      partialr2_dict[d_var][(ch_name, var)] = shap[var]
      beta_ci_dict[(d_var, "beta")][(ch_name, var)] = m_full.params[var]
      beta_ci_dict[(d_var, "CI_lower")][(ch_name, var)] = ci.loc[var, 0]
      beta_ci_dict[(d_var, "CI_upper")][(ch_name, var)] = ci.loc[var, 1]
      beta_ci_dict[(d_var, "CI Relative Width Ratio")] = (ci.loc[var, 1] - ci.loc[var, 0]) / abs(m_full.params[var])

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)
    wald_dict[ch_name][(d_var, "F-Stat")] = float(w_res.statistic)
    wald_dict[ch_name][(d_var, "p-value")] = float(w_res.pvalue)

df_beta_ci = pd.DataFrame(beta_ci_dict)
df_fit = pd.DataFrame(fit_dict)
df_wald = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict).div(df_fit.loc["Full R2"], axis=1) * 100

### 1. ADF (Stationary), VIF (Multicollinearity)

In [2]:
print("Diagnostic Assumptions (ADF, VIF):")

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "score"): float(stat),
    ("ADF", "p-val"): float(p)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = float(vif_score)

df_adf_vif = pd.DataFrame(diagnostic_dict).T
print(df_adf_vif)

Diagnostic Assumptions (ADF, VIF):
                           ADF                     VIF
                         score         p-val     score
cpi_inflation        -9.356499  8.033058e-16  1.234972
IPI                  -4.551081  1.588167e-04  1.261073
FFR_OPR_diff         -4.739985  7.074195e-05  1.192840
VIX                  -5.706259  7.476266e-07  1.631532
UST_10Y             -11.136573  3.184346e-20  1.313627
HY_OAS               -9.374128  7.243941e-16  2.567552
USDMYR              -11.083996  4.241657e-20  1.266548
CPO                  -9.141674  2.838314e-15  1.088335
Oil                  -8.562996  8.594384e-14  1.965728
KLCI                -13.146949  1.392004e-24       NaN
financials          -10.419734  1.717087e-18       NaN
energy              -11.015117  6.184580e-20       NaN
industrial_products -10.011967  1.768936e-17       NaN
plantation          -13.869750  6.497616e-26       NaN
REITs               -14.869708  1.670545e-27       NaN
technology           -4.650375

### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [3]:
print("Model Fit:")
print(df_fit.to_string())
print("\nChannel Variance Decomposition (Contribution to R²):")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())

Model Fit:
                 KLCI  financials    energy  industrial_products  plantation     REITs  technology
Full R2      0.319039    0.265130  0.236758             0.149629    0.120091  0.192223    0.233837
Adjusted R2  0.266205    0.208114  0.177540             0.083652    0.036731  0.129551    0.174393

Channel Variance Decomposition (Contribution to R²):
                              KLCI  financials     energy  industrial_products  plantation      REITs  technology
Domestic Baseline         3.568049    0.808569  21.068981             5.608906    8.689494   5.182305    1.669388
Rate Differential         1.580626    1.700713   1.350609             6.445158   20.194765  17.596788    2.475404
Global Risk Appetite     17.104903   18.071012   3.766014             6.936830    2.426317  10.933293   32.395142
Global Credit Condition  39.702759   34.132430  12.659028            56.144094   15.515055  46.708349   52.848827
FX Pass-Through          23.042624   31.615913   0.566921           

### 5. INFERENCE & FEATURE IMPORTANCE

In [4]:
print("Channel Wald Joint Significance Tests:")
print(df_wald.to_string())

Channel Wald Joint Significance Tests:
                              KLCI           financials              energy           industrial_products           plantation               REITs           technology          
                            F-Stat   p-value     F-Stat   p-value    F-Stat   p-value              F-Stat   p-value     F-Stat   p-value    F-Stat   p-value     F-Stat   p-value
Domestic Baseline         0.815139  0.445097   0.540492  0.583921  3.822282  0.024684            0.108585  0.897194   1.125086  0.328911  1.004556  0.369370   0.631290  0.533723
Rate Differential         0.758576  0.385574   0.054131  0.816435  0.393508  0.531694            1.553475  0.215134   2.858509  0.094170  2.323309  0.130170   0.005171  0.942797
Global Risk Appetite      1.018504  0.314975   0.917134  0.340220  0.991331  0.321490            0.194119  0.660331   0.001598  0.968194  0.150067  0.699181   1.910952  0.169513
Global Credit Condition   4.998505  0.008273   4.089267  0.019225  0.84

In [5]:
df_beta_ci.index.names = df_partialr2.index.names = ['Channels', 'Variables']
df_fit.columns.name = df_partialr2.columns.name = 'Indices'

df_wald.index.name = 'Channels'
df_wald.columns.names = df_beta_ci.columns.names = ['Indices', 'Metrics']

df_adf_vif.columns.names = ["Test", "Metrics"]
df_adf_vif.index.name = 'Variables'

analysis_dfs: dict = {
   'ADF VIF Diagnostics': {
      'df': df_adf_vif,
      'title': 'Time-Series Diagnostics (ADF Stationarity & VIF)',
      'caption': 'Pre-estimation time-series diagnostics: Augmented Dickey-Fuller (ADF) unit root tests for stationarity and Variance Inflation Factor (VIF) scores for multicollinearity.'
   },

   'Model Fit': {
      'df': df_fit,
      'title': 'Sector Model Fit & Explanatory Power (R²)',
      'caption': "Overall sector model goodness-of-fit metrics, reporting unadjusted Full R² and Adjusted R² across all sector multivariate regressions."
   },

   'Variance Decomposition': {
      'df': df_partialr2,
      'title': 'Channel Variance Decomposition (Shapley-LMG % Share)',
      'caption': "Shapley-LMG variance decomposition attributing common-size explanatory power (% share of total model R²) to each macroeconomic channel across all 2ᴾ sub-model orderings."
   },

   'Wald Joint Tests': {
      'df': df_wald,
      'title': 'Macro Channel Joint Significance (Wald F-Tests)',
      'caption': "Joint Wald F-test statistics and corresponding p-values evaluating the null hypothesis (H₀) that all predictor coefficients within a given transmission channel are simultaneously zero."
   },

   'Beta & Confidence Intervals': {
      'df': df_beta_ci,
      'title': 'Beta & 95% HAC Confidence Intervals for Transmission Coefficients',
      'caption': "Point estimates (β), HAC standard errors, and 95% confidence intervals (2.5% to 97.5%) for macroeconomic predictors across KLCI and Bursa Malaysia sector models."
   },
}

with open(f"{PROCESSED_DATA_PATH}/analysis_dfs.pkl", "wb") as f:
   pickle.dump(analysis_dfs, f)